In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic

c:\Users\alexb\miniconda3\envs\gnome_BERTopic\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Getting representative documents

In [5]:
df = pd.read_csv("../../results/NLP_data_advice_fulltext.csv")
docs = df["text"].str.replace('\xa0', '', regex=False).tolist()

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    use_auth_token=False
)

embeddings = embedding_model.encode(docs, show_progress_bar=True)

topic_model = BERTopic.load(
    "../../results/NLP/BERTopic_model",
    embedding_model=embedding_model
)

doc_info = topic_model.get_document_info(docs)
topics = doc_info["Topic"].tolist()

train_idxs = np.arange(len(docs))

doc_topic = pd.DataFrame({
    "Topic": topics,
    "ID": train_idxs,
    "Document": [docs[i] for i in train_idxs]
})

topic_model._create_topic_vectors(doc_topic, embeddings[train_idxs])

repr_docs, _, _, _ = topic_model._extract_representative_docs(
    topic_model.c_tf_idf_,
    doc_topic,
    topic_model.topic_representations_,
    nr_samples=1000,
    nr_repr_docs=5
)

topic_model.representative_docs_ = repr_docs
topic_model.get_topic_info()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1207.15it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 564.97it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Topic,Count,Name,Representation,KeyBERT,MMR,Representative_Docs
0,-1,181,-1_gnome_gnomes_mushrooms_round,"[gnome, gnomes, mushrooms, round, try]","[gnomes, gnome, number mushrooms, red basket, ...","[gnomes, number mushrooms, colours, mushroom, ...",[You should try multiple things in round 1 to ...
1,0,201,0_gnome_points_gnomes_colour,"[gnome, points, gnomes, colour, hat]","[colour gnomes, colour gnome, points gnomes, g...","[colour gnomes, colour gnome, points gnomes, g...",[Be as quick as you can when choosing the gnom...
2,1,143,1_mushrooms_gnome_gnomes_colour,"[mushrooms, gnome, gnomes, colour, try]","[gnomes mushrooms, gnome mushrooms, number mus...","[gnomes mushrooms, gnome mushrooms, number mus...",[It might be difficult choosing the gnome that...
3,2,143,2_basket_red_yellow_mushrooms,"[basket, red, yellow, mushrooms, gnomes]","[gnomes yellow basket, gnomes red basket, bask...","[gnomes yellow basket, gnomes red basket, bask...",[There are eight gnomes of different colours (...
4,3,98,3_points_blue_colours_pink,"[points, blue, colours, pink, purple]","[certain colours, pick colour, colours, colors...","[certain colours, pick colour, colors, blue gr...","[You'll see the colours in pairs, e.g. red and..."
5,4,76,4_basket_red_baskets_yellow,"[basket, red, baskets, yellow, points]","[gnomes baskets, basket colours, colour basket...","[gnomes baskets, basket colours, baskets red y...",[there will be 2 baskets (red and yellow) and ...
6,5,73,5_mushrooms_colours_mushroom_gives,"[mushrooms, colours, mushroom, gives, change]","[colour gives mushrooms, colours mushrooms, mu...","[colour gives mushrooms, colours mushrooms, mu...",[Each pair of colours has one that will give y...
7,6,34,6_hats_tall_hat_taller,"[hats, tall, hat, taller, tall hats]","[hat colour, short yellow hat, tall hats, shor...","[hat colour, short yellow hat, tall hats, shor...",[The patterns appear to be inconsistent but I ...
8,7,28,7_keys_just_breaks_fingers,"[keys, just, breaks, fingers, game]","[press keys, fingers keys, keyboard, stay focu...","[press keys, fingers keys, stay focused, play,...","[As each gnome appears, tap S for the left gno..."
9,8,23,8_forest_mushrooms_gnomes_green,"[forest, mushrooms, gnomes, green, orange]","[gnomes mushrooms, mushrooms gnomes, mushrooms...","[mushrooms gnomes, mushrooms forest, mushrooms...",[There are two distinct groups of gnomes(Group...


Save representative doc

In [6]:
topic_info = topic_model.get_topic_info()
rep_docs_expanded = pd.DataFrame(topic_info["Representative_Docs"].tolist())
out = pd.concat([topic_info[["Topic"]], rep_docs_expanded], axis=1)
out.to_csv("../../results/NLP/representative_docs.csv", index=False)

Save topic summary

In [7]:
topic_summary = topic_model.get_topic_info()
topic_summary.to_csv("../../results/NLP/topic_summary.csv")

Unclassified message

In [8]:
df = pd.read_csv("../../results/NLP_data_advice_fulltext.csv")
docs = df["text"].str.replace('\xa0', '', regex=False).tolist()

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    use_auth_token=False
)

topic_model = BERTopic.load(
    "../../results/NLP/BERTopic_model",
    embedding_model=embedding_model
)

doc_info = topic_model.get_document_info(docs)
topics = doc_info["Topic"].tolist()

new_topics = topic_model.reduce_outliers(docs, topics)
topic_model.update_topics(docs, topics=new_topics)
topic_model.get_topic_info()


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 876.36it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1259.02it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 1/1 [00:00<00:00, 16.23it/s]
2026-02-25 18:12:09,985 - BERTopic - WARNIN

,Topic,Count,Name,Representation,KeyBERT,MMR,Representative_Docs
0,0,250,0_the_to_gnome_and,"[the, to, gnome, and, gnomes, you, points, of,...","[colour gnomes, colour gnome, points gnomes, g...","[colour gnomes, colour gnome, points gnomes, g...",NaN
1,1,195,1_the_to_mushrooms_of,"[the, to, mushrooms, of, you, gnome, and, gnom...","[gnomes mushrooms, gnome mushrooms, number mus...","[gnomes mushrooms, gnome mushrooms, number mus...",NaN
2,2,157,2_basket_the_to_red,"[basket, the, to, red, yellow, you, and, gnome...","[gnomes yellow basket, gnomes red basket, bask...","[gnomes yellow basket, gnomes red basket, bask...",NaN
3,3,113,3_the_and_to_it,"[the, and, to, it, you, points, that, blue, fo...","[certain colours, pick colour, colours, colors...","[certain colours, pick colour, colors, blue gr...",NaN
4,4,82,4_the_basket_to_and,"[the, basket, to, and, red, yellow, gnomes, yo...","[gnomes baskets, basket colours, colour basket...","[gnomes baskets, basket colours, baskets red y...",NaN
5,5,90,5_the_to_mushrooms_you,"[the, to, mushrooms, you, and, it, of, colours...","[colour gives mushrooms, colours mushrooms, mu...","[colour gives mushrooms, colours mushrooms, mu...",NaN
6,6,39,6_hats_the_hat_and,"[hats, the, hat, and, tall, to, you, if, for, of]","[hat colour, short yellow hat, tall hats, shor...","[hat colour, short yellow hat, tall hats, shor...",NaN
7,7,47,7_the_you_and_to,"[the, you, and, to, your, as, on, it, keys, of]","[press keys, fingers keys, keyboard, stay focu...","[press keys, fingers keys, stay focused, play,...",NaN
8,8,27,8_forest_the_to_you,"[forest, the, to, you, mushrooms, other, the o...","[gnomes mushrooms, mushrooms gnomes, mushrooms...","[mushrooms gnomes, mushrooms forest, mushrooms...",NaN


save topic reduced dataframe

In [9]:
topic_distr, _ = topic_model.approximate_distribution(docs)
df_stat = pd.read_csv("../../data/NLP_data_stake.csv")

for topic_n in range(len(topic_distr[0,:])):
    topic_name = "topic_" + str(topic_n)
    df_stat[topic_name] = topic_distr[:, topic_n]

df_stat["assigned_topic"] = topic_model.topics_

df_stat.head()
parent_topic_weight = []
var_names = [f'topic_{n}' for n in range(len(topic_distr[0, :]))]

for i in range(len(df_stat["ID"])):
    parent_ID = df_stat["parent_ID"][i]
    parent_row = df_stat[df_stat["ID"] == parent_ID]
    if len(parent_row) < 1:
        parent_topic_weight.append([None for n in range(len(topic_distr[0, :]))])
    else:
        res = parent_row[var_names].values.tolist()
        parent_topic_weight.append(res[0])

parent_topic_df = pd.DataFrame(parent_topic_weight)
parent_topic_df.columns = [f'parent_topic_{n}' for n in range(len(topic_distr[0, :]))]

df_stat = pd.concat([df_stat, parent_topic_df], axis=1)
df_stat = df_stat.loc[:, ~df_stat.columns.str.contains('^Unnamed')]

df_stat.to_csv("../../results/NLP/data_topic_weights_reduced.csv", header=True, index=False)

100%|██████████| 1/1 [00:00<00:00,  1.77it/s]
